# Technical Lesson Connecting to SQL Databases

### Introduction

Now that you've gotten a brief introduction to SQL, it's time to get some hands-on practice connecting to a database and executing some queries. In this walkthough, we will connect to a SQLite (one of the flavors of SQL databases) database. Databases seldom have easy direct interfaces with the scripting language we are planning to use, so we need some way to connect to those databases for analysis.

It is very rare that the data you’ll want to use lives in a flat csv file or spreadsheet. More likely is that there will be a database file somewhere on the computer, or a database existing somewhere on the network, or in the cloud. So as data scientists we need to be able to connect to these databases in a number of ways. 


### Connecting to the Database Using Only sqlite3

We will use the sqlite3 module. The way that this module works is that we start by opening a connection to the database with the sqlite3.connect() method:

In [1]:
# Import module
import sqlite3

# Define connection as variable 
conn = sqlite3.connect('data.sqlite')

In [ ]:


#practice
import sqlite3

conn = sqlite3.connect("data.sqlite")

Here we have created a connection object within our notebook, and assigned it to ‘conn’. Let's continue on and create a cursor with which we can interact with connection.

A cursor object is what can actually execute SQL commands. You create it by calling .cursor() on the connection.

In [ ]:
# Define cursor object
cur = conn.cursor()

In [ ]:
#practice
cur = conn.cursor()

### Step 1: Viewing the List of Tables

Let's use the cursor to find out what tables are contained in this database. This requires two steps:
- Executing the query: .execute()
- Fetching the results: .fetchone(), .fetchmany(), or .fetchall()

This is because some SQL commands (e.g. deleting data) do not require results to be fetched, just commands to be executed. So the interface only fetches the results if you ask for them.


In [ ]:
# Execute the query
# (This is a special query for finding the table names. You don't need to memorize it.)

# SQLite databases all have a sqlite_master table that stores the schema information
cur.execute("""SELECT name FROM sqlite_master WHERE type = 'table';""")

# Fetch the result and store it in table_names
table_names = cur.fetchall()
table_names


In [ ]:
#practice
#cur.execute("""SELECT * FROM sqlite_master WHERE type = 'table';""")
#cur.execute("""SELECT * FROM sqlite_master""")

cur.execute("""SELECT name FROM sqlite_master WHERE type = "table" """)

table_names = cur.fetchall()
table_names

### Step 2: Selecting All Data from the `offices` Tables

If we want to get all information about the offices table, we might do something like this (* means all columns):

In [ ]:
# Execute query
cur.execute("""SELECT * FROM offices;""")
cur.fetchall()

In [ ]:
#parctice

cur.execute("""SELECT * FROM payments;""" ).fetchall()

Because .execute() returns the cursor object, it also possible to combine the previous two lines into one line, like so:

In [ ]:
# One liner
cur.execute("""SELECT * FROM offices;""").fetchall()

Information about the column names can be retrieved from the cursor. Since the most recent query was SELECT * FROM offices; the cursor will contain information about the offices table:

In [ ]:
cur.description

In [ ]:
cur.execute("""SELECT country FROM offices;""").fetchall()

In [ ]:
#practice
cur.execute("""SELECT country FROM offices;""" ).fetchall()

In [ ]:
cur.execute("""SELECT country, territory FROM offices;""").fetchall()

In [ ]:
cur.execute("""SELECT country, territory FROM offices WHERE state = "CA";""").fetchall()

This data is very usable and accessible, since it's a list of tuples rather than one giant string. If you wanted to select the phone number for the first office, for example, that would just be [0] [2] tacked on to the end of the previous Python statement.

### Step 3: Viewing the Column Names from the `offices` Table

Information about the column names can be retrieved from the cursor. Since the most recent query was `SELECT * FROM offices` the cursor will contain information about the `offices` table:

In [ ]:
# Access description of most recent query
cur.description

If we wanted to combine the previous two steps together to make a dataframe with the right column names, that would look like this:

In [ ]:
import pandas as pd

pd.DataFrame(
    data=cur.execute("""SELECT * FROM offices;""").fetchall(),
    columns=[x[0] for x in cur.description]
)

### Disconnecting from the Database
Now that we have all of the information we need, we can close the connection to the database. It is always important to close out any connections when finished just like it is important to close any open files. We don’t want to accidentally manipulate/change/delete anything within the database.


In [ ]:
# Close the connection
conn.close()

### Connecting to a Database Using sqlite3 + pandas

This is the most straightforward technique for writing SQL queries within a Python context. We can skip over the use of the cursor via built-in pandas methods.

### Step 4: Connecting to Database (again)

This is the same as the previous process, except you only need the connection, not the cursor. Pandas will read, execute, and fetch based on the provided query.

In [5]:
# Just need the connection
import sqlite3
import pandas as pd

conn = sqlite3.connect("data.sqlite")

### Step 5: Viewing the List of Tables

Now that we have the connection, we can use the **pd.read_sql** method. Instead of using the cursor, all you need is the connection variable and pandas will create a dataframe object based on the query:


In [6]:
# Provide query and connection
df = pd.read_sql("""SELECT name FROM sqlite_master WHERE type = 'table';""", conn)
df

,name
0,orderdetails
1,payments
2,offices
3,customers
4,orders
5,productlines
6,products
7,employees


As you can see, this technique created a pandas dataframe as the result of the query, rather than a string or a collection of tuples. This tends to be very convenient for future analysis.

### Step 6: Selecting all Data from the offices Table and Viewing the Column Names

Here we have combined two steps into one! **pd.read_sql** automatically retrieves the relevant column names when you select data from a table.

In [15]:
# Provide query and connection
df_all = pd.read_sql("SELECT * FROM offices;", conn)
df_all

,officeCode,city,phone,addressLine1,addressLine2,state,country,postalCode,territory
0,1,San Francisco,+1 650 219 4782,100 Market Street,Suite 300,CA,USA,94080,NA
1,2,Boston,+1 215 837 0825,1550 Court Place,Suite 102,MA,USA,02107,NA
2,3,NYC,+1 212 555 3000,523 East 53rd Street,apt. 5A,NY,USA,10022,NA
3,4,Paris,+33 14 723 4404,43 Rue Jouffroy D'abbans,,,France,75017,EMEA
4,5,Tokyo,+81 33 224 5000,4-1 Kioicho,,Chiyoda-Ku,Japan,102-8578,Japan
5,6,Sydney,+61 2 9264 2451,5-11 Wentworth Avenue,Floor #2,,Australia,NSW 2010,APAC
6,7,London,+44 20 7877 2041,25 Old Broad Street,Level 7,,UK,EC2N 1HN,EMEA


In [17]:
#practice
customer_df = pd.read_sql("""SELECT * FROM customers WHERE creditLimit > 30000""", conn)
select_customer_df = pd.read_sql(""" 
                                SELECT customerName, phone, contactFirstName, creditLimit 
                                 FROM customers
                                 WHERE creditLimit > 20000 
                                 ORDER BY creditLimit DESC 
                                """, conn)
customer_df
select_customer_df

,customerName,phone,contactFirstName,creditLimit
0,Suominen Souveniers,+358 9 8045 555,Kalle,98800.00
1,Handji Gifts& Co,+65 224 1555,Wendy,97900.00
2,Herkku Gifts,+47 2267 3215,Veysel,96800.00
3,"Toys of Finland, Co.",90-224 8555,Matti,96500.00
4,Super Scale Inc.,2035559545,Leslie,95400.00
...,...,...,...,...
117,Warburg Exchange,0241-039123,Sven,0.00
118,"Anton Designs, Ltd.",+34 913 728555,Carmen,0.00
119,Mit Vergnügen & Co.,0621-08555,Hanna,0.00
120,"Kremlin Collectables, Co.",+7 812 293 0521,Alexander,0.00


In [18]:
df_all.columns

Index(['officeCode', 'city', 'phone', 'addressLine1', 'addressLine2', 'state',
       'country', 'postalCode', 'territory'],
      dtype='str')

It is still useful to be aware of the cursor construct in case you ever need to develop Python code that fetches one result at a time, or is a command other than SELECT. But in general if you know that the end result is creating a pandas dataframe to display the result, you don't really need to interface with the cursor directly.

### Step 7: Disconnecting from the Database

This is the same as the process for using sqlite3 without pandas.

In [ ]:
conn.close()

In [20]:
conn.close()